In [1]:
!pip install numpy
!pip install pandas
!pip install requests
!pip install folium

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
import folium
import requests
import pandas

In [ ]:
arrest_table = pandas.read_csv('/content/drive/My Drive/DATA602/BPD_Arrests.csv')
arrest_table = arrest_table[pandas.notnull(arrest_table["Location 1"])]
arrest_table["lat"], arrest_table["long"] = arrest_table["Location 1"].str.split(',' , expand=True)[0], arrest_table["Location 1"].str.split(',' , expand=True)[1]
arrest_table["lat"] = arrest_table["lat"].astype(str).str.replace("(", "").astype(float)
arrest_table["long"] = arrest_table["long"].astype(str).str.replace(")", "").astype(float)
arrest_table.head()

In [ ]:
arrest_table = arrest_table.dropna(subset=['lat', 'long'])

# Initialize a map centered at the mean location
map = folium.Map(location=[arrest_table['lat'].mean(), arrest_table['long'].mean()], zoom_start=12)

# Markers with circles colored by gender
for _, row in arrest_table.iterrows():
    color = 'blue' if row['sex'] == 'M' else 'pink'  # Assign colors based on gender
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        tooltip=f"Race: {row['race']}, Age: {row['age']}, Sex: {row['sex']}"
    ).add_to(map)

map.save('gender_map.html')
map

This map visualizes the spatial distribution of arrests in a given area, with blue dots representing male arrests and pink dots representing female arrests.

The map is densely populated with blue dots, indicating a higher number of male arrests compared to female arrests across the area.

This visualization highlights a gender disparity in arrest patterns, with males being arrested at a much higher rate than females. This could be reflective of demographic, behavioral, or systemic factors within the area.


In [ ]:
race_shapes = {
    'B': {'icon': 'circle', 'color': 'beige'},
    'W': {'icon': 'square', 'color': 'green'},
    'A': {'icon': 'star', 'color': 'lightred'},
    'H': {'icon': 'triangle', 'color': 'lightblue'},
}

race_map = folium.Map(location=[arrest_table['lat'].mean(), arrest_table['long'].mean()], zoom_start=12)

# Markers for each row based on race
for _, row in arrest_table.iterrows():
    race = row['race']
    if race in race_shapes:
        icon_shape = race_shapes[race]['icon']
        color = race_shapes[race]['color']
        folium.Marker(
            location=[row['lat'], row['long']],
            icon=folium.Icon(color=color, icon=icon_shape, prefix='fa'),
            popup=f"Race: {row['race']}, Sex: {row['sex']}, Age: {row['age']}"
        ).add_to(race_map)
    else:
        # Default marker for undefined races
        folium.Marker(
            location=[row['lat'], row['long']],
            popup=f"Race: {row['race']}, Sex: {row['sex']}, Age: {row['age']}"
        ).add_to(race_map)

race_map.save("race_map.html")
race_map

his interactive data map displays arrests in Baltimore, with markers representing individual arrests. Each marker's color corresponds to a specific racial group:

**Beige**: Black

**Green**: White

**Red**: Asian

**Blue**: Hispanic

The majority of the arrests on the map are represented by beige markers, indicating a high number of arrests among Black individuals.

Green markers are visible but less prevalent compared to beige markers. They appear scattered or clustered in specific regions, possibly reflecting residential or economic disparities in those areas.

Red and blue markers are relatively sparse, indicating fewer arrests of Asian and Hispanic individuals. These groups may constitute a smaller proportion of Baltimore's population



In [ ]:
neighborhood_counts = arrest_table.groupby('neighborhood').size()
neighborhood_arrest_map = folium.Map(location=[arrest_table['lat'].mean(), arrest_table['long'].mean()], zoom_start=12)

for neighborhood, count in neighborhood_counts.items():
    neighborhood_data = arrest_table[arrest_table['neighborhood'] == neighborhood]
    lat, long = neighborhood_data[['lat', 'long']].mean()
    folium.Circle(
        location=[lat, long],
        radius=count,
        color='red',
        fill=True,
        fill_opacity=0.2,
        popup=f"{neighborhood}: {count} arrests"
    ).add_to(neighborhood_arrest_map)

neighborhood_arrest_map.save("neighborhood_arrest_map.html")
neighborhood_arrest_map

This map represents the spatial distribution and density of arrests across neighborhoods, visualized with red circles of varying sizes.

Larger circles indicate neighborhoods with higher numbers of arrests, while smaller circles represent areas with fewer arrests.

Central neighborhoods like "Downtown" and “Sandtown-Winchester” show a high concentration of arrests (3221 and 2705 arrests respectively), indicating possible hotspots of criminal activity.

Suburban and outlying areas show significantly fewer arrests, suggesting lower enforcement activity or lower crime rates.


In [ ]:

age_map = folium.Map(location=[arrest_table['lat'].mean(), arrest_table['long'].mean()], zoom_start=12)

def age_group(age):
    if age < 18:
        return "Under 18"
    elif age <= 25:
        return "18-25"
    elif age <= 40:
        return "26-40"
    else:
        return "40+"

arrest_table['age_group'] = arrest_table['age'].apply(age_group)

age_colors = {
    "Under 18": "purple",
    "18-25": "green",
    "26-40": "blue",
    "40+": "orange",
}

for _, row in arrest_table.iterrows():
    color = age_colors[row['age_group']]
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=5 + (row['age'] / 10),  # Scale radius slightly with age
        color=color,
        fill=True,
        fill_opacity=0.6,
        popup=f"Age: {row['age']}<br>Age Group: {row['age_group']}<br>Charge: {row['chargeDescription']}"
    ).add_to(age_map)
age_map.save("age_map.html")
age_map

This map appears to display arrest data by age group in Baltimore. Each point represents an individual arrest, color-coded as follows:

**Purple**: Under 18 years old

**Green**: Ages 18-25

**Blue**: Ages 26-40

**Orange**: Ages 40+

There is a dense clustering of arrests in the central and eastern parts of the mapped area, which could indicate areas of higher police activity or higher crime rates.

**Blue (26-40)** and **Green (18-25)** points seem most prevalent, suggesting that individuals aged 18-40 are the most frequently arrested.

**Orange (40+)** is also fairly common but seems less dense than the younger groups.

**Purple (Under 18)** appears sparse in comparison, which might reflect lower arrest rates among minors.
